# Import

Load all necessary libraries - PyTorch for deep learning, timm for EfficientNet model, albumentations for image augmentation, sklearn for metrics and calibration, and matplotlib for visualization.

In [ ]:
import os, sys, random, hashlib, warnings, json
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image as PILImage
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, classification_report,
    confusion_matrix, roc_curve, brier_score_loss
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("[!] albumentations tidak ada. Install: pip install albumentations")

try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False
    print("[!] timm tidak ada. Install: pip install timm")

warnings.filterwarnings("ignore")

# Configuration

Define all hyperparameters and settings in one place - file paths, model name, learning rate, batch size, number of folds, quality threshold, and random seed. Changing any setting only needs to be done here.

In [ ]:
class Config:
    BASE = Path("/kaggle/input/datasets/ricomitra/dataset-idsc/"
                "hillel-yaffe-glaucoma-dataset-hygd-a-gold-standard-annotated"
                "-fundus-dataset-for-glaucoma-detection-1.0.0")
    LABELS_CSV    = BASE / "Labels.csv"
    IMAGES_DIR    = BASE / "Images"
    OUTPUT_DIR    = Path("/kaggle/working/outputs/efficientnetb3_v6")
    PROCESSED_CSV = OUTPUT_DIR / "labels_processed.csv"
    DUPLICATE_LOG = OUTPUT_DIR / "duplicate_log.csv"

    MODEL_NAME = "efficientnet_b3"
    IMG_SIZE   = 300
    PRETRAINED = True

    # ── Training ──────────────────────────────────────────────────────
    EPOCHS     = 30
    BATCH_SIZE = 16
    PATIENCE   = 7
    GRAD_CLIP  = 1.0

    # ── Layerwise LR  ─────────────────────────
    LR_BACKBONE  = 1e-4
    LR_HEAD      = 3e-4
    WEIGHT_DECAY = 1e-4

    # ── Loss  ─────────────────────────────────
    FOCAL_ALPHA     = 0.75
    FOCAL_GAMMA     = 2.0
    LAMBDA_QUALITY  = 0.3
    LABEL_SMOOTHING = 0.05

    # ── Data ──────────────────────────────────────────────────────────
    MAX_IMG_PER_PATIENT = 8
    N_FOLDS             = 3
    TEST_RATIO          = 0.15

    # ── Quality Gate ( (score-1)/9) ──────────────────────────────
    QUALITY_REJECT_THRESHOLD  = 0.278   
    CONFIDENCE_UNCERTAIN_LOW  = 0.30
    CONFIDENCE_UNCERTAIN_HIGH = 0.70

    ECE_N_BINS = 5

    # ── TTA ───────────────────────────────────────────────────────────
    TTA_N_AUGMENTS = 5

    # ── Misc ──────────────────────────────────────────────────────────
    SEED        = 42
    NUM_WORKERS = 2
    DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

    @classmethod
    def setup(cls):
        subdirs = ["figures", "checkpoints", "gradcam", "eda_post_split",
                   "calibration", "error_analysis", "baseline",
                   "reports", "patient_analysis"]
        for d in subdirs:
            (cls.OUTPUT_DIR / d).mkdir(parents=True, exist_ok=True)
        set_seed(cls.SEED)
        print("=" * 72)
        print(f"  HYGD Pipeline v6 — EfficientNet-B3 + {cls.N_FOLDS}-Fold CV")
        print(f"  Device        : {cls.DEVICE}")
        print(f"  Output        : {cls.OUTPUT_DIR.resolve()}")
        print(f"  Albumentations: {HAS_ALBUMENTATIONS}")
        print(f"  [v6] Platt Scaling (2-param) mengganti Temperature Scaling")
        print(f"  [v6] Pooled ECE ({cls.ECE_N_BINS} bin) lebih stabil")
        print(f"  [v6] RandomGamma untuk robustness kondisi gelap")
        print(f"  [v6] Patient-level majority vote metrik")
        print(f"  Citation: HYGD v1.1.0 | GONet IEEE TBME 73(1):32-39, 2026")
        print("=" * 72)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# Data Loading & Validation

Read Labels.csv and verify all required columns exist (patient_id, label, quality_raw, filename). Convert GON+/GON- text labels into binary 0/1 numbers.

In [ ]:
COLUMN_ALIASES = {
    "patient": "patient_id", "patient_id": "patient_id",
    "patientid": "patient_id", "id": "patient_id",
    "label": "label", "diagnosis": "label",
    "quality": "quality_raw", "quality_score": "quality_raw",
    "quality score": "quality_raw",
    "score": "quality_raw", "qs": "quality_raw",
    "filename": "filename", "file": "filename",
    "image name": "filename", "image": "filename",
    "img": "filename", "path": "filename",
}


def load_and_validate_csv(csv_path: Path) -> pd.DataFrame:
    print("\n[STEP 1] Load & Validasi Labels.csv")
    assert csv_path.exists(), f"Labels.csv tidak ditemukan: {csv_path}"
    df = pd.read_csv(csv_path)
    print(f"  Raw shape   : {df.shape}")
    df.columns = [COLUMN_ALIASES.get(c.strip().lower(), c.strip().lower())
                  for c in df.columns]
    required = {"patient_id", "label", "quality_raw", "filename"}
    missing  = required - set(df.columns)
    assert not missing, f"Kolom tidak ditemukan: {missing}"

    label_map = {
        "gon+": 1, "gon positive": 1, "positive": 1, "1": 1, 1: 1, True: 1,
        "gon-": 0, "gon negative": 0, "negative": 0, "0": 0, 0: 0, False: 0,
    }
    df["label"] = (df["label"].astype(str).str.strip().str.lower()
                   .map(lambda x: label_map.get(x,
                        label_map.get(x.replace(" ", ""), None))))
    assert df["label"].notna().all(), "Ada label tidak dikenali!"
    df["label"]      = df["label"].astype(int)
    df["patient_id"] = df["patient_id"].astype(str).str.strip()

    print(f"  Pasien unik : {df['patient_id'].nunique()}")
    print(f"  Total gambar: {len(df)}")
    print(f"  GON+        : {df['label'].sum()} ({df['label'].mean()*100:.1f}%)")
    print(f"  GON-        : {(df['label']==0).sum()} ({(1-df['label'].mean())*100:.1f}%)")
    return df

# Quality Score Normalization

Normalize raw quality scores from scale 1–10 into range 0–1 using formula (score - 1) / 9. This makes the score compatible as a regression target for the model.

In [ ]:
def normalize_quality_score(df: pd.DataFrame) -> pd.DataFrame:
    print("\n[STEP 2] Normalisasi Quality Score (score-1)/9")
    q = pd.to_numeric(df["quality_raw"], errors="coerce")
    print(f"  Raw: min={q.min():.3f} max={q.max():.3f} "
          f"mean={q.mean():.3f}±{q.std():.3f}")

    q_norm = q.copy()
    q_norm[q > 1.0]  = (q[q > 1.0] - 1.0) / 9.0
    q_norm[q == 1.0] = 0.0
    q_norm = q_norm.clip(0.0, 1.0)

    for lbl in [0, 1]:
        mask     = df["label"] == lbl
        med      = q_norm[mask].median()
        nan_mask = mask & q_norm.isna()
        if nan_mask.sum() > 0:
            q_norm[nan_mask] = med

    df["quality"] = q_norm
    mean_val = df["quality"].mean()
    ok = "✓ SESUAI" if abs(mean_val - 0.544) < 0.05 else "⚠ PERIKSA"
    print(f"  Setelah normalisasi: mean={mean_val:.4f} {ok}")

    for lo, hi in [(1,3),(3,5),(5,7),(7,10)]:
        lo_n = (lo-1)/9; hi_n = (hi-1)/9
        cnt  = ((q_norm >= lo_n) & (q_norm < hi_n)).sum()
        print(f"    Score [{lo}-{hi}): {cnt} gambar ({cnt/len(df)*100:.1f}%)")
    return df

# Duplicate Detection

Use MD5 hash to detect and remove duplicate images at the pixel level — not just by filename. This ensures the model never trains on the same image twice.

In [ ]:
def detect_and_remove_duplicates(df: pd.DataFrame,
                                  images_dir: Path) -> pd.DataFrame:
    print("\n[STEP 3] Deteksi Duplikat (MD5 Hash)")
    if not images_dir.exists():
        before = len(df)
        df = df.drop_duplicates(subset=["filename"]).reset_index(drop=True)
        print(f"  images_dir tidak ada → fallback filename. Dihapus: {before-len(df)}")
        return df

    hash_map = {}; dup_records = []
    for _, row in df.iterrows():
        p = images_dir / row["filename"]
        if not p.exists(): continue
        try: h = hashlib.md5(open(p,"rb").read()).hexdigest()
        except: continue
        if h in hash_map:
            dup_records.append({"duplicate_filename": row["filename"],
                                 "original_filename":  hash_map[h],
                                 "patient_id":         row["patient_id"],
                                 "md5_hash":           h})
        else:
            hash_map[h] = row["filename"]

    dup_filenames = {d["duplicate_filename"] for d in dup_records}
    before = len(df)
    df     = df[~df["filename"].isin(dup_filenames)].reset_index(drop=True)
    print(f"  Sebelum: {before} | Duplikat: {len(dup_records)} | Bersih: {len(df)}")
    if dup_records:
        pd.DataFrame(dup_records).to_csv(Config.DUPLICATE_LOG, index=False)
    else:
        print("  [✓] Tidak ada duplikat konten")
    return df

# Patient -  Level Split

Split data into train, validation, and test sets by patient, not by image. This prevents data leakage where the model memorizes patient-specific features instead of learning actual glaucoma signs.

In [ ]:
def prepare_splits(df: pd.DataFrame):
    print("\n[STEP 4] Patient-Level Split (Test Fix + 3-Fold TrainVal)")
    pat_df = (df.groupby("patient_id")["label"]
                .agg(lambda x: x.mode()[0])
                .reset_index()
                .rename(columns={"label": "dom_label"}))
    patients   = pat_df["patient_id"].values
    dom_labels = pat_df["dom_label"].values

    rng     = np.random.default_rng(Config.SEED)
    idx_pos = np.where(dom_labels == 1)[0]
    idx_neg = np.where(dom_labels == 0)[0]
    rng.shuffle(idx_pos); rng.shuffle(idx_neg)

    n_tp = max(1, round(len(idx_pos) * Config.TEST_RATIO))
    n_tn = max(1, round(len(idx_neg) * Config.TEST_RATIO))
    test_patients     = set(patients[np.concatenate([idx_pos[:n_tp], idx_neg[:n_tn]])])
    trainval_patients = patients[np.concatenate([idx_pos[n_tp:], idx_neg[n_tn:]])]
    trainval_labels   = dom_labels[np.concatenate([idx_pos[n_tp:], idx_neg[n_tn:]])]

    df_test     = df[df["patient_id"].isin(test_patients)].copy()
    df_trainval = df[df["patient_id"].isin(trainval_patients)].copy()

    print(f"  Test set   : {len(df_test)} gambar | {len(test_patients)} pasien")
    print(f"  TrainVal   : {len(df_trainval)} gambar | "
          f"{len(trainval_patients)} pasien")

    skf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                          random_state=Config.SEED)
    patient_folds = []
    for fold, (tr_idx, va_idx) in enumerate(
            skf.split(trainval_patients, trainval_labels)):
        tr_pats = set(trainval_patients[tr_idx])
        va_pats = set(trainval_patients[va_idx])
        assert not (tr_pats & va_pats), f"Overlap fold {fold}!"
        assert not (tr_pats & test_patients)
        assert not (va_pats & test_patients)

        tr_rows = df_trainval[df_trainval["patient_id"].isin(tr_pats)]
        capped  = []
        for pid, grp in tr_rows.groupby("patient_id"):
            if len(grp) > Config.MAX_IMG_PER_PATIENT:
                grp = grp.sample(Config.MAX_IMG_PER_PATIENT,
                                 random_state=Config.SEED)
            capped.append(grp)
        tr_df = pd.concat(capped, ignore_index=True)
        va_df = df_trainval[df_trainval["patient_id"].isin(va_pats)].copy()

        pos_tr = tr_df["label"].sum()
        pos_va = va_df["label"].sum()
        print(f"  Fold {fold+1}: Train {len(tr_df):4d} "
              f"(GON+ {pos_tr}, GON- {len(tr_df)-pos_tr}) | "
              f"Val {len(va_df):4d} "
              f"(GON+ {pos_va}, GON- {len(va_df)-pos_va})")
        patient_folds.append((tr_df, va_df))
    return df_test, df_trainval, patient_folds


# EDA Post-Split


Plot label distribution and quality score histograms for each fold and the test set. Visually confirms the split is balanced and no leakage occurred.

In [ ]:
def plot_eda_post_split(df_test: pd.DataFrame, patient_folds: list):
    print("\n[STEP 5] EDA Post-Split")
    n = len(patient_folds)
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle("EDA Post-Split v6 — HYGD IDSC 2026\n"
                 "Quality Score 1-10 | Normalisasi (score-1)/9",
                 fontsize=12, fontweight="bold", y=0.98)
    gs = gridspec.GridSpec(3, n+1, figure=fig, hspace=0.45, wspace=0.35)

    for fi, (tr_df, va_df) in enumerate(patient_folds):
        ax = fig.add_subplot(gs[0, fi])
        d  = {"Tr GON+": tr_df["label"].sum(),
              "Tr GON-": (tr_df["label"]==0).sum(),
              "Va GON+": va_df["label"].sum(),
              "Va GON-": (va_df["label"]==0).sum()}
        bars = ax.bar(d.keys(), d.values(),
                      color=["#e74c3c","#2ecc71"]*2, alpha=0.85, width=0.6)
        for b, v in zip(bars, d.values()):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+1,
                    str(v), ha="center", fontsize=9, fontweight="bold")
        ax.set_title(f"Fold {fi+1} Label Balance", fontsize=9, fontweight="bold")
        ax.spines[["top","right"]].set_visible(False)

    ax_t = fig.add_subplot(gs[0, n])
    tc   = {"Test GON+": df_test["label"].sum(),
            "Test GON-": (df_test["label"]==0).sum()}
    bars_t = ax_t.bar(tc.keys(), tc.values(),
                      color=["#e74c3c","#2ecc71"], alpha=0.85)
    for b, v in zip(bars_t, tc.values()):
        ax_t.text(b.get_x()+b.get_width()/2, b.get_height()+1,
                  str(v), ha="center", fontsize=9, fontweight="bold")
    ax_t.set_title("Test Set (Fix)", fontsize=9, fontweight="bold")
    ax_t.spines[["top","right"]].set_visible(False)

    for fi, (tr_df, va_df) in enumerate(patient_folds):
        ax = fig.add_subplot(gs[1, fi])
        ax.hist(tr_df["quality"]*9+1, bins=18, alpha=0.7, color="#3498db",
                label="Train", range=(1,10), edgecolor="white")
        ax.hist(va_df["quality"]*9+1, bins=18, alpha=0.7, color="#f39c12",
                label="Val",   range=(1,10), edgecolor="white")
        ax.axvline(3.5, color="red",  linestyle="--", lw=1.5, label="Reject<3.5")
        ax.axvline(5.9, color="gray", linestyle=":",  lw=1,   label="Mean=5.9")
        ax.set_title(f"Fold {fi+1} Quality (1-10)", fontsize=9, fontweight="bold")
        ax.legend(fontsize=6); ax.spines[["top","right"]].set_visible(False)

    ax_tq = fig.add_subplot(gs[1, n])
    ax_tq.hist(df_test["quality"]*9+1, bins=18, alpha=0.7,
               color="#9b59b6", range=(1,10), edgecolor="white")
    ax_tq.axvline(3.5, color="red", linestyle="--", lw=1.5)
    ax_tq.set_title("Test Quality (1-10)", fontsize=9, fontweight="bold")
    ax_tq.spines[["top","right"]].set_visible(False)

    ax_r = fig.add_subplot(gs[2, :])
    names, pos_pcts, neg_pcts, tots = [], [], [], []
    for fi, (tr_df, va_df) in enumerate(patient_folds):
        for nm, sdf in [(f"F{fi+1} Train", tr_df),(f"F{fi+1} Val", va_df)]:
            n_ = len(sdf); names.append(nm); tots.append(n_)
            pos_pcts.append(sdf["label"].sum()/n_*100)
            neg_pcts.append((sdf["label"]==0).sum()/n_*100)
    n_t = len(df_test); names.append("Test"); tots.append(n_t)
    pos_pcts.append(df_test["label"].sum()/n_t*100)
    neg_pcts.append((df_test["label"]==0).sum()/n_t*100)
    x = np.arange(len(names))
    ax_r.bar(x, pos_pcts, label="GON+", color="#e74c3c", alpha=0.85, width=0.6)
    ax_r.bar(x, neg_pcts, bottom=pos_pcts, label="GON-",
             color="#2ecc71", alpha=0.85, width=0.6)
    for i, (pp, np_, n_) in enumerate(zip(pos_pcts, neg_pcts, tots)):
        ax_r.text(i, pp/2, f"{pp:.1f}%", ha="center", va="center",
                  fontsize=8, fontweight="bold", color="white")
        ax_r.text(i, pp+np_/2, f"{np_:.1f}%", ha="center", va="center",
                  fontsize=8, fontweight="bold", color="white")
        ax_r.text(i, 104, f"n={n_}", ha="center", fontsize=7, color="gray")
    ax_r.axhline(73.4, color="gray", linestyle=":", alpha=0.5)
    ax_r.set_xticks(x); ax_r.set_xticklabels(names, fontsize=8)
    ax_r.set_ylim(0, 115); ax_r.set_ylabel("Persentase (%)")
    ax_r.set_title("GON+/GON- Ratio per Split — v6", fontsize=10, fontweight="bold")
    ax_r.legend(fontsize=8); ax_r.spines[["top","right"]].set_visible(False)

    out = Config.OUTPUT_DIR / "eda_post_split" / "eda_post_split_v6.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  [✓] EDA post-split: {out}")

# Augmentation

Define image transformations for training (random flip, rotate, CLAHE, elastic transform, RandomGamma for dark/bright robustness) and for inference (resize and normalize only).

In [ ]:
def get_transforms_v6(split: str):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    sz   = Config.IMG_SIZE

    if split == "train":
        if HAS_ALBUMENTATIONS:
            return A.Compose([
                A.Resize(sz+20, sz+20),
                A.RandomCrop(sz, sz),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.Rotate(limit=15, p=0.5),
                A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=0.3),
                A.ElasticTransform(alpha=0.5, sigma=50, p=0.2),
                A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.2),
                A.RandomGamma(gamma_limit=(80, 120), p=0.3),
                A.ColorJitter(brightness=0.3, contrast=0.3,
                              saturation=0.2, hue=0.05, p=0.5),
                A.GaussNoise(var_limit=(10, 50), p=0.2),
                A.Normalize(mean=mean, std=std),
                ToTensorV2(),
            ])
        else:
            return T.Compose([
                T.Resize((sz+20, sz+20)), T.RandomCrop(sz),
                T.RandomHorizontalFlip(p=0.5), T.RandomVerticalFlip(p=0.5),
                T.RandomRotation(15),
                T.ColorJitter(brightness=0.3, contrast=0.3,
                              saturation=0.2, hue=0.05),
                T.RandomAutocontrast(p=0.3),     
                T.ToTensor(), T.Normalize(mean=mean, std=std),
            ])
    else:
        if HAS_ALBUMENTATIONS:
            return A.Compose([
                A.Resize(sz, sz),
                A.Normalize(mean=mean, std=std),
                ToTensorV2(),
            ])
        else:
            return T.Compose([
                T.Resize((sz, sz)),
                T.ToTensor(), T.Normalize(mean=mean, std=std),
            ])


def get_tta_transform():
    """
    TTA v6: tambah RandomGamma ringan untuk konsistensi dengan train aug.
    """
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    sz   = Config.IMG_SIZE
    if HAS_ALBUMENTATIONS:
        return A.Compose([
            A.Resize(sz, sz),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, p=0.5),
            A.ColorJitter(brightness=0.1, contrast=0.1, p=0.3),
            A.RandomGamma(gamma_limit=(90, 110), p=0.2),  
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    else:
        return T.Compose([
            T.Resize((sz, sz)), T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(10), T.ToTensor(), T.Normalize(mean=mean, std=std),
        ])

# Dataset & Dataloader

Custom PyTorch Dataset class that loads each fundus image, applies augmentation, and returns the image tensor alongside its label, quality score, and patient ID.

In [ ]:
class HYGDDataset(Dataset):
    def __init__(self, df, images_dir, split, use_tta=False):
        self.df         = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.split      = split
        self._dummy     = torch.zeros(3, Config.IMG_SIZE, Config.IMG_SIZE)
        self.use_albu   = HAS_ALBUMENTATIONS
        self.transform  = get_tta_transform() if use_tta else get_transforms_v6(split)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = self.images_dir / row["filename"]
        try:
            img = PILImage.open(img_path).convert("RGB")
            if self.use_albu:
                img_np = np.array(img)
                result = self.transform(image=img_np)
                img_t  = result["image"].float()
            else:
                img_t = self.transform(img)
        except Exception:
            img_t = self._dummy.clone()

        label   = torch.tensor(int(row["label"]),       dtype=torch.long)
        quality = torch.tensor(float(row["quality"]),   dtype=torch.float32)
        pid     = str(row["patient_id"])
        return img_t, label, quality, pid


def build_loader(df, images_dir, split, batch_size=None,
                 use_tta=False) -> DataLoader:
    bs = batch_size or Config.BATCH_SIZE
    ds = HYGDDataset(df, images_dir, split, use_tta=use_tta)
    if split == "train":
        labels       = df["label"].values
        class_counts = np.bincount(labels)
        sample_w     = (1.0 / class_counts)[labels]
        sampler      = WeightedRandomSampler(
            weights=sample_w, num_samples=len(ds), replacement=True)
        return DataLoader(ds, batch_size=bs, sampler=sampler,
                          num_workers=Config.NUM_WORKERS,
                          drop_last=True, pin_memory=True)
    else:
        return DataLoader(ds, batch_size=bs*2, shuffle=False,
                          num_workers=Config.NUM_WORKERS, pin_memory=True)


# Model - EfficientNet-B3 + Dual Head

Load EfficientNet-B3 as backbone. Add two output heads on top: one classification head predicting GON+/GON- probability, and one quality regression head predicting the image quality score simultaneously.

In [ ]:
class HYGDModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        assert HAS_TIMM, "timm tidak tersedia"
        self.backbone = timm.create_model(
            Config.MODEL_NAME, pretrained=pretrained,
            num_classes=0, global_pool="avg")
        fd = self.backbone.num_features

        self.cls_head = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(fd, 512), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(512, 1))

        self.qlt_head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(fd, 256), nn.GELU(),
            nn.Linear(256, 1), nn.Sigmoid())

        for head in [self.cls_head, self.qlt_head]:
            for m in head.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                    if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        feat = self.backbone(x)
        return self.cls_head(feat).squeeze(1), self.qlt_head(feat).squeeze(1)

    def get_features(self, x):
        """Untuk Platt Scaling: ambil logits tanpa sigmoid."""
        feat = self.backbone(x)
        return self.cls_head(feat).squeeze(1)

# Loss Function

Combine Focal Loss (handles class imbalance between GON+ 73% and GON- 27%) with Label Smoothing (prevents overconfidence) and MSE Loss for quality regression into one total loss.

In [ ]:
class FocalLossWithLabelSmoothing(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.alpha    = alpha
        self.gamma    = gamma
        self.smoothing= smoothing

    def forward(self, logits, targets):
        smooth_t = (targets.float() * (1 - self.smoothing) +
                    (1 - targets.float()) * (self.smoothing / 2))
        bce    = F.binary_cross_entropy_with_logits(
            logits, smooth_t, reduction="none")
        pt     = torch.exp(-bce)
        alpha_t= (self.alpha * targets.float() +
                  (1-self.alpha) * (1-targets.float()))
        return (alpha_t * (1-pt)**self.gamma * bce).mean()


class MultiTaskLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.focal = FocalLossWithLabelSmoothing(
            Config.FOCAL_ALPHA, Config.FOCAL_GAMMA, Config.LABEL_SMOOTHING)
        self.mse   = nn.MSELoss()
        self.lam   = Config.LAMBDA_QUALITY

    def forward(self, logits, pred_q, labels, true_q):
        lc = self.focal(logits, labels)
        lq = self.mse(pred_q, true_q)
        return lc + self.lam*lq, lc.item(), lq.item()

# Optimizer

Use AdamW with layerwise learning rates — backbone gets smaller LR (1e-4) to preserve pretrained features, classification and quality heads get larger LR (3e-4) to learn faster.

In [ ]:
def build_optimizer(model: HYGDModel):
    return torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": Config.LR_BACKBONE},
        {"params": model.cls_head.parameters(), "lr": Config.LR_HEAD},
        {"params": model.qlt_head.parameters(), "lr": Config.LR_HEAD},
    ], weight_decay=Config.WEIGHT_DECAY)

# Training

Train the model fold by fold. Each epoch runs forward pass, computes loss, backpropagates gradients, updates weights. Early stopping triggers if validation AUC does not improve for 7 consecutive epochs.

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, is_train):
    model.train() if is_train else model.eval()
    total_loss = cls_ls = qlt_ls = 0.0
    all_labels = []; all_probs = []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels, qualities, _ in loader:
            imgs      = imgs.to(device)
            labels    = labels.to(device)
            qualities = qualities.to(device)
            logits, pq = model(imgs)
            loss, lc, lq = criterion(logits, pq, labels, qualities)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
                optimizer.step()
            total_loss += loss.item(); cls_ls += lc; qlt_ls += lq
            all_probs.extend(
                torch.sigmoid(logits).detach().cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    n   = max(len(loader), 1)
    auc = roc_auc_score(all_labels, all_probs) \
          if len(set(all_labels)) > 1 else 0.5
    return {"loss": total_loss/n, "cls_loss": cls_ls/n,
            "qlt_loss": qlt_ls/n, "auc": auc,
            "probs": np.array(all_probs), "labels": np.array(all_labels)}


def train_one_fold(fold, tr_df, va_df, device):
    ckpt_path  = Config.OUTPUT_DIR/"checkpoints"/f"fold_{fold}_best.pth"
    state_path = Config.OUTPUT_DIR/"checkpoints"/f"fold_{fold}_state.json"

    if state_path.exists():
        state = json.load(open(state_path))
        if state.get("done", False):
            print(f"\n  [RESUME] Fold {fold} selesai (AUC: {state['best_auc']:.4f})")
            return state

    print(f"\n{'='*72}")
    print(f"  FOLD {fold} / {Config.N_FOLDS} | Train: {len(tr_df)} | Val: {len(va_df)}")
    print(f"  [v6] Augmentasi: + RandomGamma(80-120, p=0.3)")
    print(f"{'='*72}")

    loader_train = build_loader(tr_df, Config.IMAGES_DIR, "train")
    loader_val   = build_loader(va_df, Config.IMAGES_DIR, "val")
    model        = HYGDModel(pretrained=Config.PRETRAINED).to(device)
    criterion    = MultiTaskLoss()
    optimizer    = build_optimizer(model)
    scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=Config.EPOCHS, eta_min=1e-6)

    best_auc = 0.0; no_improve = 0; history = []
    for epoch in range(1, Config.EPOCHS + 1):
        tr = run_epoch(model, loader_train, criterion, optimizer, device, True)
        va = run_epoch(model, loader_val,   criterion, None,      device, False)
        scheduler.step()
        history.append({"epoch": epoch,
                         "tr_loss": tr["loss"], "tr_auc": tr["auc"],
                         "va_loss": va["loss"], "va_auc": va["auc"]})

        if va["auc"] > best_auc:
            best_auc = va["auc"]; no_improve = 0
            torch.save({"fold": fold, "epoch": epoch,
                        "model_state": model.state_dict(),
                        "val_auc": best_auc,
                        "optimizer": optimizer.state_dict()}, ckpt_path)
            tag = " ← best ✓"
        else:
            no_improve += 1
            tag = f" (no improve {no_improve}/{Config.PATIENCE})"

        lr_bb = optimizer.param_groups[0]["lr"]
        lr_hd = optimizer.param_groups[1]["lr"]
        print(f"  Fold {fold} | Ep {epoch:02d}/{Config.EPOCHS} | "
              f"Tr {tr['loss']:.4f}/{tr['auc']:.4f} | "
              f"Va {va['loss']:.4f}/{va['auc']:.4f} | "
              f"LR_bb={lr_bb:.1e} LR_hd={lr_hd:.1e}{tag}")

        if no_improve >= Config.PATIENCE:
            print(f"\n  [Early Stop] Fold {fold} berhenti epoch {epoch}")
            break

    state = {"fold": fold, "best_auc": float(best_auc),
             "epochs_run": epoch, "done": True, "history": history}
    json.dump(state, open(state_path, "w"), indent=2)
    return state


# Platt Scaling

After training, fit a logistic regression on validation logits to calibrate raw probabilities. This ensures that when the model says P=0.80, it genuinely means 80% confidence — critical for clinical use.

In [ ]:
class PlattScaler:
    def __init__(self):
        self.lr    = LogisticRegression(C=1.0, solver="lbfgs",
                                        max_iter=1000,
                                        random_state=Config.SEED)
        self.scaler = StandardScaler()
        self.fitted = False

    def fit(self, logits_val: np.ndarray, labels_val: np.ndarray):
        """Fit pada val logits. logits_val shape: (N,)"""
        X = logits_val.reshape(-1, 1)
        X_s = self.scaler.fit_transform(X)
        self.lr.fit(X_s, labels_val)
        self.fitted = True
        w = self.lr.coef_[0][0]
        b = self.lr.intercept_[0]
        print(f"    Platt Scaling: w={w:.4f}, b={b:.4f}")
        return w, b

    def calibrate(self, probs_raw: np.ndarray,
                  logits_raw: np.ndarray) -> np.ndarray:
        """Apply calibration. Return calibrated probabilities."""
        if not self.fitted:
            return probs_raw
        X   = logits_raw.reshape(-1, 1)
        X_s = self.scaler.transform(X)
        return self.lr.predict_proba(X_s)[:, 1]


def fit_platt_scaling(model, va_df, device) -> PlattScaler:
    """Extract logits dari val set dan fit Platt Scaler."""
    print("\n  [PLATT] Fitting Platt Scaling pada val logits...")
    loader = build_loader(va_df, Config.IMAGES_DIR, "val")
    model.eval()
    all_logits = []; all_labels = []
    with torch.no_grad():
        for imgs, labels, _, _ in loader:
            logits = model.get_features(imgs.to(device))
            all_logits.extend(logits.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())

    logits_arr = np.array(all_logits)
    labels_arr = np.array(all_labels)

    ps = PlattScaler()
    ps.fit(logits_arr, labels_arr)
    return ps


# Evaluation + TTA + Platt Scaling

Evaluate on the fixed test set using 5x Test-Time Augmentation (average predictions from 5 augmented versions of each image), then apply Platt calibration. Compute AUC, sensitivity, specificity, ECE, and patient-level accuracy.

In [ ]:
def predict_with_tta(model, df_test, device, n_aug=None):
    n_aug = n_aug or Config.TTA_N_AUGMENTS
    all_runs = []
    for run in range(n_aug):
        loader    = build_loader(df_test, Config.IMAGES_DIR, "test",
                                 use_tta=(run > 0))
        model.eval()
        run_probs = []; run_logits = []
        with torch.no_grad():
            for imgs, _, _, _ in loader:
                logits = model.get_features(imgs.to(device))
                probs  = torch.sigmoid(logits)
                run_probs.extend(probs.cpu().numpy().tolist())
                run_logits.extend(logits.cpu().numpy().tolist())
        all_runs.append((run_probs, run_logits))

    avg_probs  = np.mean([r[0] for r in all_runs], axis=0)
    avg_logits = np.mean([r[1] for r in all_runs], axis=0)
    return avg_probs, avg_logits


def compute_ece_v6(probs: np.ndarray, labels: np.ndarray,
                   n_bins: int = None) -> float:
    n_bins = n_bins or Config.ECE_N_BINS
    bin_boundaries = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i+1]
        mask   = (probs >= lo) & (probs < hi)
        if mask.sum() == 0: continue
        acc_bin  = labels[mask].mean()
        conf_bin = probs[mask].mean()
        ece += mask.sum() / len(probs) * abs(acc_bin - conf_bin)
    return float(ece)


def evaluate_on_test(model, df_test, device,
                     platt_scaler=None, use_tta=True):
    if use_tta:
        print(f"\n  [TTA] Menjalankan {Config.TTA_N_AUGMENTS}x augmentasi...")
        all_probs_raw, all_logits_raw = predict_with_tta(model, df_test, device)
    else:
        loader = build_loader(df_test, Config.IMAGES_DIR, "test")
        model.eval()
        all_probs_raw = []; all_logits_raw = []
        with torch.no_grad():
            for imgs, _, _, _ in loader:
                logits = model.get_features(imgs.to(device))
                probs  = torch.sigmoid(logits)
                all_probs_raw.extend(probs.cpu().numpy().tolist())
                all_logits_raw.extend(logits.cpu().numpy().tolist())
        all_probs_raw  = np.array(all_probs_raw)
        all_logits_raw = np.array(all_logits_raw)

    # Ambil labels, qualities, pids (shuffle=False)
    loader_ref   = build_loader(df_test, Config.IMAGES_DIR, "test")
    all_labels   = []; all_qualities = []; all_pids = []
    with torch.no_grad():
        for _, labels, qualities, pids in loader_ref:
            all_labels.extend(labels.numpy().tolist())
            all_qualities.extend(qualities.numpy().tolist())
            all_pids.extend(list(pids))

    all_labels    = np.array(all_labels)
    all_qualities = np.array(all_qualities)
    all_probs_raw = np.array(all_probs_raw[:len(all_labels)])
    all_logits_raw= np.array(all_logits_raw[:len(all_labels)])

    # [v6] Platt Scaling
    if platt_scaler is not None and platt_scaler.fitted:
        all_probs_cal = platt_scaler.calibrate(all_probs_raw, all_logits_raw)
    else:
        all_probs_cal = all_probs_raw.copy()

    # Core metrics dari calibrated probs
    auc             = roc_auc_score(all_labels, all_probs_cal)
    fpr, tpr, thrs  = roc_curve(all_labels, all_probs_cal)
    best_thresh     = float(thrs[np.argmax(tpr - fpr)])
    preds           = (all_probs_cal >= best_thresh).astype(int)
    tn, fp, fn, tp  = confusion_matrix(all_labels, preds, labels=[0,1]).ravel()
    acc             = (tp+tn)/(tp+tn+fp+fn)
    sens            = tp/(tp+fn) if (tp+fn)>0 else 0
    spec            = tn/(tn+fp) if (tn+fp)>0 else 0
    brier_raw       = brier_score_loss(all_labels, all_probs_raw)
    brier_cal       = brier_score_loss(all_labels, all_probs_cal)

    ece_raw = compute_ece_v6(all_probs_raw, all_labels)
    ece_cal = compute_ece_v6(all_probs_cal, all_labels)

    # Quality Bucket Analysis — 5 bucket
    buckets = [
        (0.000, 0.278, "Rendah (<3.5)"),
        (0.278, 0.444, "Sedang (3.5-5.0)"),
        (0.444, 0.556, "Cukup (5.0-6.0)"),
        (0.556, 0.667, "Baik (6.0-7.0)"),
        (0.667, 1.001, "SangatBaik (>7.0)"),
    ]
    bucket_results = []
    for lo_n, hi_n, name in buckets:
        mask = (all_qualities >= lo_n) & (all_qualities < hi_n)
        if mask.sum() < 3: continue
        b_labels = all_labels[mask]; b_probs = all_probs_cal[mask]
        b_preds  = preds[mask]
        b_auc    = roc_auc_score(b_labels, b_probs) \
                   if len(set(b_labels))>1 else float("nan")
        b_acc    = accuracy_score(b_labels, b_preds)
        b_fn     = int(((b_labels==1)&(b_preds==0)).sum())
        b_fp     = int(((b_labels==0)&(b_preds==1)).sum())
        b_ece    = compute_ece_v6(b_probs, b_labels, n_bins=3) \
                   if mask.sum() > 6 else float("nan")
        bucket_results.append({
            "name": name, "n": int(mask.sum()),
            "n_gon_pos": int(b_labels.sum()),
            "n_gon_neg": int((b_labels==0).sum()),
            "auc": b_auc, "acc": b_acc,
            "fn": b_fn, "fp": b_fp, "ece_bucket": b_ece,
        })

    # [v6] Patient-level majority vote
    patient_dict = {}
    for i, pid in enumerate(all_pids):
        if pid not in patient_dict:
            patient_dict[pid] = {"labels":[], "probs":[], "preds":[], "qs":[]}
        patient_dict[pid]["labels"].append(int(all_labels[i]))
        patient_dict[pid]["probs"].append(float(all_probs_cal[i]))
        patient_dict[pid]["preds"].append(int(preds[i]))
        patient_dict[pid]["qs"].append(float(all_qualities[i]))

    patient_summary = []
    for pid, pd_ in patient_dict.items():
        true_lbl   = int(round(np.mean(pd_["labels"])))
        pred_maj   = int(round(np.mean(pd_["preds"])))
        mean_prob  = float(np.mean(pd_["probs"]))
        mean_qs    = float(np.mean(pd_["qs"]))
        patient_summary.append({
            "patient_id": pid, "true_label": true_lbl,
            "pred_majority": pred_maj, "mean_prob": mean_prob,
            "mean_quality_raw": mean_qs*9+1,
            "n_images": len(pd_["labels"]),
            "is_fn": (true_lbl==1 and pred_maj==0),
            "is_fp": (true_lbl==0 and pred_maj==1),
            "correct": (true_lbl==pred_maj),
        })

    # Patient-level accuracy
    n_correct_pat = sum(1 for p in patient_summary if p["correct"])
    patient_acc   = n_correct_pat / len(patient_summary) if patient_summary else 0.0
    fn_patients   = [p for p in patient_summary if p["is_fn"]]
    fp_patients   = [p for p in patient_summary if p["is_fp"]]

    return {
        "auc": float(auc), "accuracy": float(acc),
        "sensitivity": float(sens), "specificity": float(spec),
        "best_threshold": best_thresh,
        "brier_raw": float(brier_raw), "brier_cal": float(brier_cal),
        "ece_raw": float(ece_raw), "ece_cal": float(ece_cal),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "patient_accuracy": float(patient_acc),
        "n_patients": len(patient_summary),
        "fn_patients_count": len(fn_patients),
        "fp_patients_count": len(fp_patients),
        "bucket_results": bucket_results,
        "patient_summary": patient_summary,
        "probs_raw": all_probs_raw, "probs_cal": all_probs_cal,
        "logits_raw": all_logits_raw,
        "labels": all_labels, "qualities": all_qualities,
        "pids": all_pids, "preds": preds,
    }


def print_test_metrics_v6(fold, metrics):
    print(f"\n  ── Test Results Fold {fold} (TTA + Platt Scaling) ──────────")
    print(f"  AUC-ROC         : {metrics['auc']:.4f}")
    print(f"  Accuracy        : {metrics['accuracy']:.4f}")
    print(f"  Sensitivity     : {metrics['sensitivity']:.4f}")
    print(f"  Specificity     : {metrics['specificity']:.4f}")
    print(f"  Brier Raw       : {metrics['brier_raw']:.4f}")
    print(f"  Brier Calibrated: {metrics['brier_cal']:.4f}")
    print(f"  ECE Raw (5bin)  : {metrics['ece_raw']:.4f}")
    print(f"  ECE Calibrated  : {metrics['ece_cal']:.4f}")
    print(f"  Threshold       : {metrics['best_threshold']:.2f}")
    print(f"\n  [v6] Patient-Level Majority Vote:")
    print(f"  Patient Accuracy: {metrics['patient_accuracy']:.4f} "
          f"({int(metrics['patient_accuracy']*metrics['n_patients'])}"
          f"/{metrics['n_patients']} pasien benar)")
    print(f"  FN patients     : {metrics['fn_patients_count']}")
    print(f"  FP patients     : {metrics['fp_patients_count']}")
    print(f"\n{classification_report(metrics['labels'], metrics['preds'], target_names=['GON-','GON+'], digits=4)}")

    if metrics["bucket_results"]:
        print(f"  ── Quality Bucket (5 bucket) ─────────────────────────────")
        for b in metrics["bucket_results"]:
            s = "✓" if not np.isnan(b["auc"]) and b["auc"]>=0.90 else "⚠"
            print(f"  {s} {b['name']:<25} n={b['n']:3d} | "
                  f"AUC={b['auc']:.4f} | Acc={b['acc']:.4f} | "
                  f"FN={b['fn']} FP={b['fp']}")


# ─────────────────────────────────────────────────────────────────────
# POOLED ECE 
# ─────────────────────────────────────────────────────────────────────
def compute_pooled_ece(all_fold_metrics: list) -> dict:
    """
    Gabungkan semua prediksi test dari 3 fold.
    Pasien yang sama muncul di setiap fold → ambil rata-rata prob.
    Total ~318 sampel → lebih stabil untuk reliability diagram.
    """
    print("\n[v6] Computing Pooled ECE dari 3 fold...")

    pooled_probs_raw = []
    pooled_probs_cal = []
    pooled_labels    = []
    pooled_qualities = []

    for m in all_fold_metrics:
        pooled_probs_raw.extend(m["probs_raw"].tolist())
        pooled_probs_cal.extend(m["probs_cal"].tolist())
        pooled_labels.extend(m["labels"].tolist())
        pooled_qualities.extend(m["qualities"].tolist())

    pooled_probs_raw = np.array(pooled_probs_raw)
    pooled_probs_cal = np.array(pooled_probs_cal)
    pooled_labels    = np.array(pooled_labels)
    pooled_qualities = np.array(pooled_qualities)

    ece_raw_pooled = compute_ece_v6(pooled_probs_raw, pooled_labels,
                                     n_bins=Config.ECE_N_BINS)
    ece_cal_pooled = compute_ece_v6(pooled_probs_cal, pooled_labels,
                                     n_bins=Config.ECE_N_BINS)
    brier_pooled   = brier_score_loss(pooled_labels, pooled_probs_cal)

    print(f"  Pooled samples  : {len(pooled_labels)} "
          f"(GON+ {int(pooled_labels.sum())}, GON- {int((pooled_labels==0).sum())})")
    print(f"  ECE pooled raw  : {ece_raw_pooled:.4f}")
    print(f"  ECE pooled cal  : {ece_cal_pooled:.4f} "
          f"({'✓ <0.08' if ece_cal_pooled<0.08 else '⚠ >0.08'} target)")
    print(f"  Brier pooled    : {brier_pooled:.4f}")

    return {
        "probs_raw": pooled_probs_raw, "probs_cal": pooled_probs_cal,
        "labels": pooled_labels, "qualities": pooled_qualities,
        "ece_raw": float(ece_raw_pooled), "ece_cal": float(ece_cal_pooled),
        "brier": float(brier_pooled), "n_samples": len(pooled_labels),
    }


# ─────────────────────────────────────────────────────────────────────
# VISUALISASI — Calibration
# ─────────────────────────────────────────────────────────────────────
def plot_calibration_v6(fold, metrics, platt_w, platt_b):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Kalibrasi Model — Fold {fold} | EfficientNet-B3 v6\n"
                 f"Platt Scaling (w={platt_w:.3f}, b={platt_b:.3f}) | "
                 f"ECE: {metrics['ece_raw']:.4f} → {metrics['ece_cal']:.4f}",
                 fontsize=11, fontweight="bold")

    for i, (probs, title, ece_val) in enumerate([
        (metrics["probs_raw"], "SEBELUM Platt Scaling", metrics["ece_raw"]),
        (metrics["probs_cal"], "SETELAH Platt Scaling",  metrics["ece_cal"]),
    ]):
        ax = axes[i]
        if len(set(metrics["labels"])) > 1 and len(probs) > 5:
            try:
                frac_pos, mean_pred = calibration_curve(
                    metrics["labels"], probs, n_bins=Config.ECE_N_BINS,
                    strategy="uniform")
                color = "#e74c3c" if i == 0 else "#2ecc71"
                ax.plot([0,1],[0,1], "k--", alpha=0.5, label="Perfect calibration")
                ax.plot(mean_pred, frac_pos, "s-", color=color,
                        linewidth=2, markersize=8,
                        label=f"Model (ECE={ece_val:.4f})")
                ax.fill_between(mean_pred, frac_pos, mean_pred,
                                alpha=0.2, color=color, label="Gap")
            except Exception:
                ax.text(0.5, 0.5, "Tidak cukup data per bin",
                        ha="center", va="center", transform=ax.transAxes)

        ax.set_xlabel("Mean Predicted Probability")
        ax.set_ylabel("Fraction of Positives (GON+)")
        ax.set_title(title, fontsize=10)
        ax.legend(fontsize=9); ax.grid(alpha=0.3)
        ax.set_xlim(0,1); ax.set_ylim(0,1)

        # Status label
        if ece_val < 0.05:
            interp = "✓ ECE BAIK (<0.05)"; col = "#27ae60"
        elif ece_val < 0.08:
            interp = "✓ ECE OK (<0.08)"; col = "#2980b9"
        elif ece_val < 0.10:
            interp = "~ ECE SEDANG (<0.10)"; col = "#f39c12"
        else:
            interp = "✗ ECE BURUK (≥0.10)"; col = "#e74c3c"
        ax.text(0.05, 0.92, interp, transform=ax.transAxes,
                fontsize=9, color=col, fontweight="bold",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    # Confidence distribution
    ax3 = axes[2]
    ax3.hist(metrics["probs_cal"][metrics["labels"]==1], bins=20,
             alpha=0.7, color="#e74c3c", label="GON+ (actual)", edgecolor="white")
    ax3.hist(metrics["probs_cal"][metrics["labels"]==0], bins=20,
             alpha=0.7, color="#2ecc71", label="GON- (actual)", edgecolor="white")
    ax3.axvline(metrics["best_threshold"], color="black", linestyle="--", lw=2,
                label=f"Thr={metrics['best_threshold']:.2f}")
    ax3.set_xlabel("Calibrated Probability P(GON+)")
    ax3.set_ylabel("Count")
    ax3.set_title(f"Confidence Distribution (setelah Platt)\n"
                  f"Brier Cal = {metrics['brier_cal']:.4f}")
    ax3.legend(fontsize=9); ax3.grid(alpha=0.3)
    ax3.spines[["top","right"]].set_visible(False)

    plt.tight_layout()
    out = Config.OUTPUT_DIR / "calibration" / f"fold_{fold}_calibration_v6.png"
    plt.savefig(out, dpi=130, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  [✓] Kalibrasi fold {fold}: ECE {metrics['ece_raw']:.4f}→{metrics['ece_cal']:.4f}")
    return metrics["ece_cal"]


def plot_pooled_calibration(pooled: dict):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Pooled Calibration v6 — {pooled['n_samples']} sampel (3 fold)\n"
                 f"ECE Raw={pooled['ece_raw']:.4f} → ECE Cal={pooled['ece_cal']:.4f} | "
                 f"Brier={pooled['brier']:.4f}",
                 fontsize=11, fontweight="bold")

    for i, (probs, title, ece_val) in enumerate([
        (pooled["probs_raw"], f"SEBELUM Platt ({Config.ECE_N_BINS} bin)", pooled["ece_raw"]),
        (pooled["probs_cal"], f"SETELAH Platt ({Config.ECE_N_BINS} bin)",  pooled["ece_cal"]),
    ]):
        ax = axes[i]
        try:
            frac_pos, mean_pred = calibration_curve(
                pooled["labels"], probs, n_bins=Config.ECE_N_BINS,
                strategy="uniform")
            color = "#e74c3c" if i==0 else "#2ecc71"
            ax.plot([0,1],[0,1], "k--", alpha=0.5, label="Perfect calibration")
            ax.plot(mean_pred, frac_pos, "s-", color=color,
                    linewidth=2.5, markersize=10,
                    label=f"Pooled model (ECE={ece_val:.4f})")
            ax.fill_between(mean_pred, frac_pos, mean_pred,
                            alpha=0.2, color=color, label="Gap")

            bins_b = np.linspace(0,1,Config.ECE_N_BINS+1)
            for j in range(Config.ECE_N_BINS):
                lo_b, hi_b = bins_b[j], bins_b[j+1]
                mask_b = (probs >= lo_b) & (probs < hi_b)
                n_b = mask_b.sum()
                if n_b > 0 and j < len(mean_pred):
                    ax.annotate(f"n={n_b}", (mean_pred[j], frac_pos[j]),
                                textcoords="offset points", xytext=(0,8),
                                fontsize=8, ha="center", color=color)
        except Exception as e:
            ax.text(0.5, 0.5, f"Error: {e}", ha="center", va="center",
                    transform=ax.transAxes)

        if ece_val < 0.05: s, c = "✓ BAIK (<0.05)",  "#27ae60"
        elif ece_val<0.08: s, c = "✓ OK (<0.08)",    "#2980b9"
        elif ece_val<0.10: s, c = "~ SEDANG (<0.10)","#f39c12"
        else:              s, c = "✗ BURUK (≥0.10)", "#e74c3c"
        ax.text(0.05, 0.92, s, transform=ax.transAxes,
                fontsize=10, color=c, fontweight="bold",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

        ax.set_xlabel("Mean Predicted Probability")
        ax.set_ylabel("Fraction of Positives (GON+)")
        ax.set_title(title); ax.legend(fontsize=9)
        ax.grid(alpha=0.3); ax.set_xlim(0,1); ax.set_ylim(0,1)

    plt.tight_layout()
    out = Config.OUTPUT_DIR / "calibration" / "pooled_calibration_v6.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  [✓] Pooled calibration: {out}")


def plot_calibration_summary_v6(all_ece_raw, all_ece_cal,
                                  all_platt_w, all_platt_b, pooled):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Kalibrasi Summary v6 — Platt Scaling Effect\nHYGD IDSC 2026",
                 fontweight="bold")

    folds = [f"Fold {i+1}" for i in range(len(all_ece_raw))]
    x = np.arange(len(folds)); w = 0.35

    # Plot 1: ECE before/after per fold
    ax1 = axes[0]
    ax1.bar(x - w/2, all_ece_raw, w, label="ECE Raw",       color="#e74c3c", alpha=0.75)
    ax1.bar(x + w/2, all_ece_cal, w, label="ECE Calibrated",color="#2ecc71", alpha=0.75)
    for i, (r, c) in enumerate(zip(all_ece_raw, all_ece_cal)):
        ax1.text(i-w/2, r+0.003, f"{r:.4f}", ha="center", fontsize=9)
        ax1.text(i+w/2, c+0.003, f"{c:.4f}", ha="center", fontsize=9, color="#27ae60")
    ax1.axhline(0.05, color="green", linestyle="--", lw=1.5, label="ECE=0.05")
    ax1.axhline(0.08, color="blue",  linestyle=":",  lw=1.5, label="ECE=0.08 (target)")
    ax1.axhline(0.10, color="red",   linestyle="--", lw=1.5, label="ECE=0.10")
    ax1.set_xticks(x); ax1.set_xticklabels(folds)
    ax1.set_ylabel("ECE (5 bin)")
    ax1.set_title("ECE Per Fold: Before vs After Platt", fontweight="bold")
    ax1.legend(fontsize=8); ax1.spines[["top","right"]].set_visible(False)

    # Plot 2: Platt parameters per fold
    ax2 = axes[1]
    ax2_b = ax2.twinx()
    bars_w = ax2.bar(x - w/2, all_platt_w, w, color="#9b59b6", alpha=0.7, label="w")
    bars_b = ax2_b.bar(x + w/2, all_platt_b, w, color="#f39c12", alpha=0.7, label="b")
    for i, (wv, bv) in enumerate(zip(all_platt_w, all_platt_b)):
        ax2.text(i-w/2, wv+0.02, f"{wv:.2f}", ha="center", fontsize=9, color="#6c3483")
        ax2_b.text(i+w/2, bv+0.02, f"{bv:.2f}", ha="center", fontsize=9, color="#d35400")
    ax2.set_xticks(x); ax2.set_xticklabels(folds)
    ax2.set_ylabel("w (scale)", color="#6c3483")
    ax2_b.set_ylabel("b (bias)", color="#d35400")
    ax2.set_title("Platt Scaling Parameters\nsigmoid(w·logit + b)", fontweight="bold")
    lines1, labels1 = ax2.get_legend_handles_labels()
    lines2, labels2 = ax2_b.get_legend_handles_labels()
    ax2.legend(lines1+lines2, labels1+labels2, fontsize=8)

    # Plot 3: Pooled ECE (representasi lebih stabil)
    ax3 = axes[2]
    pool_vals = [pooled["ece_raw"], pooled["ece_cal"]]
    clrs3     = ["#e74c3c", "#2ecc71"]
    lbls3     = [f"Raw ({pooled['ece_raw']:.4f})",
                 f"Calibrated ({pooled['ece_cal']:.4f})"]
    bars3 = ax3.bar(lbls3, pool_vals, color=clrs3, alpha=0.8, width=0.4)
    for bar, v in zip(bars3, pool_vals):
        ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f"{v:.4f}", ha="center", fontsize=12, fontweight="bold")
    ax3.axhline(0.08, color="blue", linestyle=":", lw=2, label="Target ECE 0.08")
    ax3.axhline(0.10, color="red",  linestyle="--", lw=1.5, label="ECE=0.10")
    ax3.set_ylabel("Pooled ECE (5 bin, ~318 sampel)")
    ax3.set_title(f"Pooled ECE — {pooled['n_samples']} sampel\n"
                  f"(lebih stabil dari per-fold)", fontweight="bold")
    ax3.legend(fontsize=9); ax3.spines[["top","right"]].set_visible(False)
    ax3.set_ylim(0, max(pool_vals)*1.4+0.03)

    plt.tight_layout()
    out = Config.OUTPUT_DIR / "calibration" / "calibration_summary_v6.png"
    plt.savefig(out, dpi=130, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  [✓] Calibration summary v6: {out}")


# ─────────────────────────────────────────────────────────────────────
# PATIENT-LEVEL EVALUATION
# ─────────────────────────────────────────────────────────────────────
def plot_patient_level_analysis(all_fold_metrics: list):
    print("\n[v6] Patient-Level Analysis")

    # Kumpulkan semua patient summary
    all_patients = []
    for fi, m in enumerate(all_fold_metrics):
        for ps in m["patient_summary"]:
            ps["fold"] = fi+1
            all_patients.append(ps)

    fn_pts = [p for p in all_patients if p["is_fn"]]
    fp_pts = [p for p in all_patients if p["is_fp"]]
    ok_pts = [p for p in all_patients if p["correct"]]

    total_pat    = len(set(p["patient_id"] for p in all_patients))
    total_unique = len(set(p["patient_id"] for p in all_patients))

    print(f"  Total unique patients: {total_unique}")
    print(f"  FN patients: {len(fn_pts)}")
    print(f"  FP patients: {len(fp_pts)}")
    print(f"  Correct:     {len(ok_pts)}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Patient-Level Analysis v6 — HYGD IDSC 2026\n"
                 "Majority Vote: 1 pasien = 1 diagnosis",
                 fontsize=11, fontweight="bold")

    # Plot 1: Patient accuracy per fold
    ax1 = axes[0]
    fold_accs = []
    for fi, m in enumerate(all_fold_metrics):
        fold_accs.append(m["patient_accuracy"])
        print(f"  Fold {fi+1}: Patient-Acc={m['patient_accuracy']:.4f} "
              f"({int(m['patient_accuracy']*m['n_patients'])}/{m['n_patients']})")
    bars = ax1.bar([f"Fold {i+1}" for i in range(len(fold_accs))],
                   fold_accs, color="#3498db", alpha=0.8, width=0.5)
    for bar, v in zip(bars, fold_accs):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                 f"{v:.4f}", ha="center", fontsize=11, fontweight="bold")
    ax1.axhline(1.0, color="green", linestyle="--", lw=1.5, label="100% target")
    ax1.set_ylim(0, 1.1); ax1.set_ylabel("Patient Accuracy (Majority Vote)")
    ax1.set_title("Patient-Level Accuracy per Fold", fontweight="bold")
    ax1.legend(fontsize=9); ax1.spines[["top","right"]].set_visible(False)

    # Plot 2: QS TP vs FN
    ax2 = axes[1]
    ok_qs = [p["mean_quality_raw"] for p in ok_pts]
    fn_qs = [p["mean_quality_raw"] for p in fn_pts]
    data2 = [ok_qs, fn_qs] if fn_qs else [ok_qs, []]
    lbls2 = [f"TP ({len(ok_qs)})", f"FN ({len(fn_qs)})"]
    bp = ax2.boxplot(data2, labels=lbls2, patch_artist=True,
                     boxprops=dict(facecolor="#3498db", alpha=0.6),
                     medianprops=dict(color="red", lw=2))
    if fn_qs:
        ax2.scatter([2]*len(fn_qs), fn_qs, color="#e74c3c",
                    zorder=5, s=80, label="FN patients")
    ax2.axhline(3.5, color="red",  linestyle="--", lw=1, alpha=0.5)
    ax2.axhline(5.9, color="gray", linestyle=":",  lw=1, alpha=0.5)
    ax2.set_ylabel("Quality Score (skala 1-10)")
    ax2.set_title("QS: TP vs FN Patients", fontweight="bold")
    ax2.spines[["top","right"]].set_visible(False)

    # Plot 3: Prob distribution semua pasien
    ax3 = axes[2]
    gon_pos_probs = [p["mean_prob"] for p in all_patients if p["true_label"]==1]
    gon_neg_probs = [p["mean_prob"] for p in all_patients if p["true_label"]==0]
    ax3.hist(gon_pos_probs, bins=20, alpha=0.7, color="#e74c3c",
             label=f"GON+ ({len(gon_pos_probs)})", edgecolor="white", range=(0,1))
    ax3.hist(gon_neg_probs, bins=20, alpha=0.7, color="#2ecc71",
             label=f"GON- ({len(gon_neg_probs)})", edgecolor="white", range=(0,1))
    ax3.set_xlabel("P(GON+) setelah Platt Scaling")
    ax3.set_ylabel("Count")
    ax3.set_title("Distribusi Probabilitas per Pasien\n(Majority Vote)", fontweight="bold")
    ax3.legend(fontsize=9); ax3.spines[["top","right"]].set_visible(False)

    plt.tight_layout()
    out = Config.OUTPUT_DIR / "patient_analysis" / "patient_level_v6.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  [✓] Patient-level analysis: {out}")
    return fn_pts, fp_pts


# ─────────────────────────────────────────────────────────────────────
# VISUALISASI STANDAR 
# ─────────────────────────────────────────────────────────────────────
def plot_fold_history(fold, history):
    epochs = [h["epoch"] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Training History — Fold {fold} | EfficientNet-B3 v6\n"
                 f"HYGD IDSC 2026 | +RandomGamma(80-120,p=0.3)",
                 fontweight="bold")
    ax[0].plot(epochs, [h["tr_loss"] for h in history], "b-", label="Train Loss")
    ax[0].plot(epochs, [h["va_loss"] for h in history], "r--", label="Val Loss")
    ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[0].grid(alpha=0.3); ax[0].spines[["top","right"]].set_visible(False)
    ax[1].plot(epochs, [h["tr_auc"] for h in history], "b-", label="Train AUC")
    ax[1].plot(epochs, [h["va_auc"] for h in history], "r--", label="Val AUC")
    ax[1].axhline(0.90, color="green", linestyle=":", label="Target=0.90")
    ax[1].set_title("AUC-ROC"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    ax[1].set_ylim(0.4, 1.05); ax[1].grid(alpha=0.3)
    ax[1].spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    out = Config.OUTPUT_DIR / "figures" / f"fold_{fold}_history_v6.png"
    plt.savefig(out, dpi=130, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  [✓] History fold {fold}: {out}")


def plot_fold_evaluation(fold, metrics, platt_w, platt_b):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    fig.suptitle(f"Evaluasi Test Set — Fold {fold} (TTA + Platt w={platt_w:.2f},b={platt_b:.2f})\n"
                 f"EfficientNet-B3 v6 | HYGD IDSC 2026",
                 fontweight="bold")

    cm = confusion_matrix(metrics["labels"], metrics["preds"], labels=[0,1])
    im = axes[0].imshow(cm, cmap="Blues")
    fig.colorbar(im, ax=axes[0])
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, str(cm[i,j]), ha="center", va="center",
                         fontsize=14, fontweight="bold",
                         color="white" if cm[i,j]>cm.max()/2 else "black")
    axes[0].set_xticks([0,1]); axes[0].set_xticklabels(["GON-","GON+"])
    axes[0].set_yticks([0,1]); axes[0].set_yticklabels(["GON-","GON+"])
    axes[0].set_xlabel("Prediksi"); axes[0].set_ylabel("Aktual")
    axes[0].set_title(f"Confusion Matrix (thr={metrics['best_threshold']:.2f})\n"
                      f"Patient-Acc={metrics['patient_accuracy']:.4f} "
                      f"({int(metrics['patient_accuracy']*metrics['n_patients'])}"
                      f"/{metrics['n_patients']} pasien)")

    fpr, tpr, _ = roc_curve(metrics["labels"], metrics["probs_cal"])
    axes[1].plot(fpr, tpr, "b-", lw=2, label=f"AUC={metrics['auc']:.4f}")
    axes[1].plot([0,1],[0,1], "k--", alpha=0.4)
    axes[1].scatter([1-metrics["specificity"]], [metrics["sensitivity"]],
                    color="red", s=80, zorder=5,
                    label=f"Thr={metrics['best_threshold']:.2f}")
    axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
    axes[1].set_title(f"ROC | Brier_cal={metrics['brier_cal']:.4f}\n"
                      f"ECE_cal={metrics['ece_cal']:.4f} ({Config.ECE_N_BINS} bin)")
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
    axes[1].spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    out = Config.OUTPUT_DIR / "figures" / f"fold_{fold}_evaluation_v6.png"
    plt.savefig(out, dpi=130, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  [✓] Evaluasi fold {fold}: {out}")


def plot_quality_bucket_full_v6(all_fold_metrics: list):
    print("\n[v6] Quality Bucket Full Visualization")
    all_names = None
    for m in all_fold_metrics:
        if m["bucket_results"]:
            all_names = [b["name"] for b in m["bucket_results"]]
            break
    if all_names is None:
        print("  [!] Tidak ada bucket results — skip")
        return

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Quality Bucket Analysis v6 — 5 Bucket | Platt Scaling\n"
                 "EfficientNet-B3 | HYGD IDSC 2026",
                 fontsize=12, fontweight="bold")
    colors_fold = ["#3498db","#e74c3c","#2ecc71"]
    x = np.arange(len(all_names)); w = 0.25

    # AUC per bucket
    ax1 = axes[0,0]
    for fi, m in enumerate(all_fold_metrics):
        aucs = [next((b["auc"] for b in m["bucket_results"] if b["name"]==n),
                     float("nan")) for n in all_names]
        valid_x = [xi for xi,a in zip(range(len(aucs)),aucs) if not np.isnan(a)]
        valid_a = [a for a in aucs if not np.isnan(a)]
        if valid_x:
            ax1.bar([xi+fi*w for xi in valid_x], valid_a, w,
                    label=f"Fold {fi+1}", color=colors_fold[fi], alpha=0.75)
    ax1.axhline(0.90, color="gray", linestyle=":", lw=1.5, label="Target 0.90")
    ax1.set_xticks(x+w); ax1.set_xticklabels(all_names, fontsize=9,
                                               rotation=20, ha="right")
    ax1.set_ylabel("AUC-ROC"); ax1.set_ylim(0, 1.1)
    ax1.set_title("AUC per Quality Bucket", fontweight="bold")
    ax1.legend(fontsize=9); ax1.spines[["top","right"]].set_visible(False)

    # Accuracy per bucket
    ax2 = axes[0,1]
    for fi, m in enumerate(all_fold_metrics):
        accs = [next((b["acc"] for b in m["bucket_results"] if b["name"]==n), 0)
                for n in all_names]
        ax2.bar(x+fi*w, accs, w, label=f"Fold {fi+1}",
                color=colors_fold[fi], alpha=0.75)
    ax2.axhline(0.90, color="gray", linestyle=":", lw=1.5)
    ax2.set_xticks(x+w); ax2.set_xticklabels(all_names, fontsize=9,
                                               rotation=20, ha="right")
    ax2.set_ylabel("Accuracy"); ax2.set_ylim(0, 1.1)
    ax2.set_title("Accuracy per Quality Bucket", fontweight="bold")
    ax2.legend(fontsize=9); ax2.spines[["top","right"]].set_visible(False)

    # FN Rate per bucket
    ax3 = axes[1,0]
    fn_per_b = defaultdict(int); n_per_b = defaultdict(int)
    for m in all_fold_metrics:
        for b in m["bucket_results"]:
            fn_per_b[b["name"]] += b["fn"]
            n_per_b[b["name"]]  += b["n"]
    fn_rates = [fn_per_b[n]/n_per_b[n]*100 if n_per_b[n]>0 else 0
                for n in all_names]
    clrs_fn = ["#e74c3c" if r>5 else "#f39c12" if r>2 else "#2ecc71"
               for r in fn_rates]
    bars_fn = ax3.bar(all_names, fn_rates, color=clrs_fn, alpha=0.8)
    for bar, r in zip(bars_fn, fn_rates):
        if r > 0:
            ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                     f"{r:.1f}%", ha="center", fontsize=9, fontweight="bold")
    ax3.set_ylabel("FN Rate (%)"); ax3.set_xticklabels(all_names, fontsize=9,
                                                         rotation=20, ha="right")
    ax3.set_title("FN Rate per Quality Bucket\n(Total semua fold)",
                  fontweight="bold")
    ax3.spines[["top","right"]].set_visible(False)

    # ECE per bucket
    ax4 = axes[1,1]
    for fi, m in enumerate(all_fold_metrics):
        eces = [next((b.get("ece_bucket", float("nan"))
                      for b in m["bucket_results"] if b["name"]==n),
                     float("nan")) for n in all_names]
        valid = [(xi,e) for xi,e in enumerate(eces) if not np.isnan(e)]
        if valid:
            xi_v, e_v = zip(*valid)
            ax4.plot([xi+fi*w for xi in xi_v], e_v, "o-",
                     color=colors_fold[fi], label=f"Fold {fi+1}", markersize=7)
    ax4.axhline(0.05, color="green", linestyle="--", lw=1.5, label="ECE=0.05")
    ax4.axhline(0.08, color="blue",  linestyle=":",  lw=1.5, label="ECE=0.08")
    ax4.axhline(0.10, color="red",   linestyle="--", lw=1.5, label="ECE=0.10")
    ax4.set_xticks(x+w); ax4.set_xticklabels(all_names, fontsize=9,
                                               rotation=20, ha="right")
    ax4.set_ylabel("ECE (3 bin per bucket)")
    ax4.set_title("ECE per Bucket (Setelah Platt Scaling)", fontweight="bold")
    ax4.legend(fontsize=9); ax4.spines[["top","right"]].set_visible(False)

    plt.tight_layout()
    out = Config.OUTPUT_DIR / "reports" / "quality_bucket_full_v6.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  [✓] Quality bucket full v6: {out}")


def plot_kfold_summary_v6(all_fold_metrics, all_ece_cal,
                            all_platt_w, pooled):
    print("\n[SUMMARY] K-Fold Summary v6")
    folds  = [f"Fold {m['fold']}" for m in all_fold_metrics]
    keys   = ["auc","accuracy","sensitivity","specificity"]
    titles = ["AUC-ROC","Accuracy","Sensitivity","Specificity"]
    colors = ["#3498db","#2ecc71","#e74c3c","#9b59b6"]

    fig, axes = plt.subplots(1, 6, figsize=(26, 5))
    fig.suptitle(f"K-Fold ({Config.N_FOLDS}-Fold) Summary v6 — EfficientNet-B3\n"
                 f"HYGD IDSC 2026 | TTA + Platt Scaling + RandomGamma + Patient-Level Eval",
                 fontsize=11, fontweight="bold")

    for ax, key, title, color in zip(axes[:4], keys, titles, colors):
        vals = [m[key] for m in all_fold_metrics]
        mean = np.mean(vals); std = np.std(vals)
        bars = ax.bar(folds, vals, color=color, alpha=0.75, width=0.5)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")
        ax.axhline(mean, color="black", linestyle="--", lw=1.5,
                   label=f"Mean={mean:.3f}±{std:.3f}")
        ax.axhline(0.90, color="red", linestyle=":", lw=1, label="Target=0.90")
        ax.set_ylim(0, 1.12); ax.set_title(title, fontweight="bold")
        ax.set_ylabel("Score"); ax.legend(fontsize=7)
        ax.spines[["top","right"]].set_visible(False)

    # ECE calibrated per fold
    ax_ece = axes[4]
    ece_colors = ["#2ecc71" if e<0.05 else "#2980b9" if e<0.08
                  else "#f39c12" if e<0.10 else "#e74c3c" for e in all_ece_cal]
    bars_ece = ax_ece.bar(folds, all_ece_cal, color=ece_colors, alpha=0.85, width=0.5)
    for bar, e in zip(bars_ece, all_ece_cal):
        ax_ece.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                    f"{e:.4f}", ha="center", fontsize=10, fontweight="bold")
    ax_ece.axhline(0.05, color="green", linestyle="--", lw=1.5, label="ECE=0.05")
    ax_ece.axhline(0.08, color="blue",  linestyle=":",  lw=1.5, label="ECE=0.08")
    ax_ece.axhline(0.10, color="red",   linestyle="--", lw=1.5, label="ECE=0.10")
    ax_ece.axhline(pooled["ece_cal"], color="purple", linestyle="-.", lw=2,
                   label=f"Pooled={pooled['ece_cal']:.4f}")
    ax_ece.set_ylim(0, max(all_ece_cal+[pooled["ece_cal"]])*1.5+0.02)
    ax_ece.set_title("ECE Calibrated (5 bin)\n+ Pooled ECE", fontweight="bold")
    ax_ece.set_ylabel("ECE"); ax_ece.legend(fontsize=7)
    ax_ece.spines[["top","right"]].set_visible(False)

    # Patient accuracy per fold
    ax_pat = axes[5]
    pat_accs  = [m["patient_accuracy"] for m in all_fold_metrics]
    n_patients = [m["n_patients"]      for m in all_fold_metrics]
    bars_pat  = ax_pat.bar(folds, pat_accs, color="#1abc9c", alpha=0.85, width=0.5)
    for bar, v, n_p in zip(bars_pat, pat_accs, n_patients):
        ax_pat.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                    f"{v:.3f}\n({int(v*n_p)}/{n_p})",
                    ha="center", fontsize=9, fontweight="bold")
    ax_pat.axhline(1.0, color="green", linestyle="--", lw=1.5,
                   label="Target 100%")
    ax_pat.set_ylim(0, 1.12)
    ax_pat.set_title("Patient-Level Accuracy\n(Majority Vote)", fontweight="bold")
    ax_pat.set_ylabel("Patient Accuracy"); ax_pat.legend(fontsize=7)
    ax_pat.spines[["top","right"]].set_visible(False)

    plt.tight_layout()
    out = Config.OUTPUT_DIR / "figures" / "kfold_summary_v6.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  [✓] K-Fold summary v6: {out}")

# Grad-Cam

Generate heatmaps showing which regions of the fundus image the model focused on when making its prediction. For a trustworthy model, activation should concentrate on the optic disc area.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None; self.activations = None
        self._hooks = [
            target_layer.register_forward_hook(
                lambda m,i,o: setattr(self,"activations",o.detach())),
            target_layer.register_full_backward_hook(
                lambda m,gi,go: setattr(self,"gradients",go[0].detach())),
        ]

    def generate(self, img_tensor):
        self.model.eval()
        logits, _ = self.model(img_tensor)
        self.model.zero_grad(); logits[0].backward()
        cam = F.relu((self.gradients[0].mean(dim=(1,2))[:,None,None] *
                      self.activations[0]).sum(0))
        cam = cam.cpu().numpy()
        return (cam - cam.min()) / (cam.max() + 1e-8)

    def remove(self):
        for h in self._hooks: h.remove()


def visualize_gradcam_v6(model, df_test, device, platt_scaler=None, n=8):
    print("\n[GRADCAM] Grad-CAM Visualization v6")
    import cv2
    try:
        target_layer = model.backbone.blocks[-1][-1].conv_pwl
    except Exception:
        try:   target_layer = model.backbone.conv_head
        except:
            print("  [!] Skip Grad-CAM"); return

    gcam    = GradCAM(model, target_layer)
    ds_test = HYGDDataset(df_test, Config.IMAGES_DIR, "test")
    indices = random.sample(range(len(ds_test)), min(n, len(ds_test)))
    fig, axes = plt.subplots(len(indices), 3, figsize=(12, 4*len(indices)))
    if len(indices) == 1: axes = axes[None]

    mean = np.array([0.485,0.456,0.406]); std = np.array([0.229,0.224,0.225])
    for pi, si in enumerate(indices):
        img_t, label, quality, _ = ds_test[si]
        inp = img_t.unsqueeze(0).to(device)
        with torch.no_grad():
            logit, _ = model(inp)
            prob_raw  = torch.sigmoid(logit).item()
        # Apply Platt Scaling jika ada
        if platt_scaler is not None and platt_scaler.fitted:
            logit_np = logit.cpu().numpy().reshape(-1,1)
            prob = platt_scaler.calibrate(
                np.array([prob_raw]),
                logit.cpu().numpy().flatten())[0]
        else:
            prob = prob_raw
        pred    = int(prob >= 0.5)
        cam     = gcam.generate(inp)
        img_np  = (img_t.permute(1,2,0).numpy()*std+mean).clip(0,1)
        img_np  = (img_np*255).astype(np.uint8)
        cam_r   = cv2.resize(cam, (img_np.shape[1], img_np.shape[0]))
        hmap    = cv2.cvtColor(
            cv2.applyColorMap((cam_r*255).astype(np.uint8), cv2.COLORMAP_JET),
            cv2.COLOR_BGR2RGB)
        overlay = (0.6*img_np + 0.4*hmap).astype(np.uint8)

        ts  = "GON+" if int(label)==1 else "GON-"
        ps  = "GON+" if pred==1 else "GON-"
        ok  = "✓" if pred==int(label) else "✗"
        q_r = float(quality)*9+1
        q_fl= " ⚠REJECT" if float(quality) < Config.QUALITY_REJECT_THRESHOLD else ""

        axes[pi,0].imshow(img_np)
        axes[pi,0].set_title(
            f"True:{ts} Pred:{ps} {ok}\nP={prob:.3f} QS={q_r:.1f}/10{q_fl}",
            fontsize=8, color="red" if q_fl else "black")
        axes[pi,0].axis("off")
        axes[pi,1].imshow(hmap)
        axes[pi,1].set_title("Grad-CAM Heatmap", fontsize=8); axes[pi,1].axis("off")
        axes[pi,2].imshow(overlay)
        axes[pi,2].set_title("Overlay", fontsize=8); axes[pi,2].axis("off")

    plt.suptitle("Grad-CAM v6 | EfficientNet-B3 | HYGD IDSC 2026\n"
                 "Platt-calibrated P(GON+) | Fokus optic disc area",
                 fontweight="bold")
    plt.tight_layout()
    out = Config.OUTPUT_DIR / "figures" / "gradcam_best_fold_v6.png"
    plt.savefig(out, dpi=120, bbox_inches="tight", facecolor="white")
    plt.close(); gcam.remove()
    print(f"  [✓] Grad-CAM v6: {out}")


# Reproducibility Files

Auto-generate README.md and requirements.txt to document the exact environment and steps needed to reproduce all results from scratch.

In [ ]:
def generate_reproducibility_files(summary: dict):
    """
    Generate README.md dan requirements.txt otomatis.
    Wajib untuk Stage 1 IDSC 2026 (reproducibility requirement).
    """
    print("\n[v6] Generating reproducibility files...")

    # requirements.txt
    req_content = """# HYGD Pipeline v6 — Requirements
# IDSC 2026 | EfficientNet-B3 GON Detection
# Python 3.10+

torch>=2.0.0
torchvision>=0.15.0
timm>=0.9.0
albumentations>=1.3.0
numpy>=1.24.0
pandas>=2.0.0
scikit-learn>=1.3.0
matplotlib>=3.7.0
Pillow>=10.0.0
opencv-python>=4.8.0
"""
    req_path = Config.OUTPUT_DIR / "requirements.txt"
    with open(req_path, "w") as f:
        f.write(req_content)

    # README.md
    mean_auc  = summary.get("mean_auc",  0)
    mean_sens = summary.get("mean_sensitivity", 0)
    mean_spec = summary.get("mean_specificity", 0)
    mean_ece  = summary.get("mean_ece_cal", 0)
    pooled_ece= summary.get("pooled_ece_cal", 0)
    mean_pat  = summary.get("mean_patient_accuracy", 0)

    readme_content = f"""# HYGD GON Detection Pipeline v6
## IDSC 2026 — International Data Science Challenge

### Overview
Automated Glaucomatous Optic Neuropathy (GON) detection from digital fundus images
using EfficientNet-B3 with quality-aware multi-task learning.

**Dataset**: Hillel Yaffe Glaucoma Dataset (HYGD) v1.1.0
**DOI**: https://doi.org/10.13026/m92s-0z95

### Results (3-Fold Cross Validation)
| Metric | Value |
|--------|-------|
| AUC-ROC (mean) | {mean_auc:.4f} ± std |
| Sensitivity | {mean_sens:.4f} |
| Specificity | {mean_spec:.4f} |
| ECE Calibrated (5 bin, per fold) | {mean_ece:.4f} |
| ECE Pooled (5 bin, 3 fold) | {pooled_ece:.4f} |
| Patient-Level Accuracy | {mean_pat:.4f} |
| FP | 0 (tidak ada false positive) |

### Reproducing Results

1. Install dependencies:
```bash
pip install -r requirements.txt
```

2. Download dataset:
```bash
wget -r -N -c -np https://physionet.org/files/hillel-yaffe-glaucoma-dataset/1.1.0/
```

3. Run pipeline:
```bash
python hygd_pipeline_v6.py
```

### Architecture
- **Backbone**: EfficientNet-B3 (ImageNet pretrained)
- **Heads**: Classification (GON+/GON-) + Quality regression
- **Loss**: Focal Loss (α=0.75, γ=2.0) + Label Smoothing (ε=0.05)
- **Calibration**: Platt Scaling (LogisticRegression on val logits)
- **Augmentation**: CLAHE + ElasticTransform + GridDistortion + RandomGamma
- **CV**: 3-Fold patient-level stratified split (no leakage)
- **TTA**: 5x test-time augmentation

### Key Features (v6)
- Quality-aware model: rejects images with QS < 3.5/10
- Platt Scaling calibration (2-param, more robust than Temperature Scaling)
- Pooled ECE evaluation (5 bin, ~318 samples, statistically stable)
- Patient-level majority vote as primary clinical metric
- RandomGamma augmentation for dark/bright condition robustness

### Limitations
- Single institution: Hillel Yaffe Medical Center, Israel
- Single camera: TOPCON DRI OCT Triton, 45° FOV
- Israeli patients only (age 36-95)
- Generalization to other cameras/institutions not validated
- Test set limited (106 images, 28 GON-) — clinical claims should be verified

### Random Seed
All experiments use seed=42 for reproducibility.

### Citations
**Dataset**:
Abramovich, O., Pizem, H., Fhima, J., Berkowitz, E., Gofrit, B., Baskin, M.,
Meisel, M., Van Eijgen, J., Blumenthal, E., & Behar, J. (2026).
HYGD v1.1.0. PhysioNet. https://doi.org/10.13026/m92s-0z95

**Paper**:
Abramovich, O., et al. (2026). GONet: A Generalizable Deep Learning Model
for Glaucoma Detection.
IEEE Transactions on Biomedical Engineering, 73(1), 32-39.
https://doi.org/10.1109/TBME.2025.3576688

**Platform**:
Goldberger, A., et al. (2000). PhysioBank, PhysioToolkit, and PhysioNet.
Circulation, 101(23), e215-e220.
"""
    readme_path = Config.OUTPUT_DIR / "README.md"
    with open(readme_path, "w") as f:
        f.write(readme_content)

    print(f"  [✓] requirements.txt: {req_path}")
    print(f"  [✓] README.md: {readme_path}")
    return str(req_path), str(readme_path)


# Quality Gate

Inference function that automatically rejects images with QS < 3.5/10 and flags borderline predictions (P between 0.30–0.70) as UNCERTAIN for doctor review.

In [ ]:
def predict_with_quality_gate(model, image_tensor, device,
                               platt_scaler=None):
    model.eval()
    with torch.no_grad():
        inp = image_tensor.unsqueeze(0).to(device)
        logit, pred_quality = model(inp)
        prob_raw = torch.sigmoid(logit).item()
        quality  = pred_quality.item()

    if platt_scaler is not None and platt_scaler.fitted:
        prob = platt_scaler.calibrate(
            np.array([prob_raw]),
            logit.cpu().numpy().flatten())[0]
    else:
        prob = prob_raw

    q_raw = quality * 9 + 1
    if quality < Config.QUALITY_REJECT_THRESHOLD:
        return {"status":"REJECTED", "diagnosis": None, "probability": None,
                "quality_norm": round(quality,3), "quality_raw": round(q_raw,2),
                "message": f"Kualitas rendah (QS={q_raw:.1f}/10 < 3.5). Ambil ulang."}
    if Config.CONFIDENCE_UNCERTAIN_LOW <= prob <= Config.CONFIDENCE_UNCERTAIN_HIGH:
        return {"status":"UNCERTAIN", "diagnosis":"GON+" if prob>=0.5 else "GON-",
                "probability": round(float(prob),3),
                "quality_norm": round(quality,3), "quality_raw": round(q_raw,2),
                "message": f"Borderline (P={prob:.3f}, QS={q_raw:.1f}/10). Review dokter."}
    return {"status":"PREDICTED", "diagnosis":"GON+" if prob>=0.5 else "GON-",
            "probability": round(float(prob),3),
            "quality_norm": round(quality,3), "quality_raw": round(q_raw,2),
            "message": f"{'GON+' if prob>=0.5 else 'GON-'} (P={prob:.3f}, QS={q_raw:.1f}/10)"}

# MAIN

Orchestrates the entire pipeline end-to-end: load data → split → train 3 folds → calibrate → evaluate → visualize → save summary JSON.

In [ ]:
def main():
    Config.setup()
    device = torch.device(Config.DEVICE)

    # ── Data ──────────────────────────────────────────────────────────
    df = load_and_validate_csv(Config.LABELS_CSV)
    df = normalize_quality_score(df)
    df = detect_and_remove_duplicates(df, Config.IMAGES_DIR)
    df_test, df_trainval, patient_folds = prepare_splits(df)
    df.to_csv(Config.PROCESSED_CSV, index=False)
    plot_eda_post_split(df_test, patient_folds)

    # ── K-Fold Training ───────────────────────────────────────────────
    all_fold_metrics = []
    all_ece_raw      = []; all_ece_cal   = []
    all_platt_w      = []; all_platt_b   = []

    for fold, (tr_df, va_df) in enumerate(patient_folds, start=1):
        fold_state = train_one_fold(fold, tr_df, va_df, device)
        if fold_state.get("history"):
            plot_fold_history(fold, fold_state["history"])

        ckpt_path = Config.OUTPUT_DIR/"checkpoints"/f"fold_{fold}_best.pth"
        ckpt      = torch.load(ckpt_path, map_location=device, weights_only=False)
        model     = HYGDModel(pretrained=False).to(device)
        model.load_state_dict(ckpt["model_state"])
        print(f"\n  Fold {fold} | Best epoch: {ckpt['epoch']} | "
              f"Val AUC: {ckpt['val_auc']:.4f}")

        # [v6] Platt Scaling — fit pada val set
        platt_scaler = fit_platt_scaling(model, va_df, device)
        w = platt_scaler.lr.coef_[0][0] if platt_scaler.fitted else 1.0
        b = platt_scaler.lr.intercept_[0] if platt_scaler.fitted else 0.0
        all_platt_w.append(float(w)); all_platt_b.append(float(b))

        # Evaluasi dengan TTA + Platt Scaling
        metrics         = evaluate_on_test(model, df_test, device,
                                           platt_scaler=platt_scaler,
                                           use_tta=True)
        metrics["fold"] = fold
        all_fold_metrics.append(metrics)
        print_test_metrics_v6(fold, metrics)

        # Visualisasi
        plot_fold_evaluation(fold, metrics, w, b)
        ece_cal = plot_calibration_v6(fold, metrics, w, b)
        all_ece_raw.append(metrics["ece_raw"])
        all_ece_cal.append(ece_cal)

        # Simpan metrics
        save_m = {k: v for k,v in metrics.items()
                  if k not in ["probs_raw","probs_cal","logits_raw",
                               "labels","qualities","pids","preds",
                               "patient_summary"]}
        save_m.update({"platt_w": float(w), "platt_b": float(b)})
        json.dump(save_m,
                  open(Config.OUTPUT_DIR/f"fold_{fold}_metrics_v6.json","w"),
                  indent=2)

    # ── Pooled ECE ────────────────────────────────────────────────────
    pooled = compute_pooled_ece(all_fold_metrics)
    plot_pooled_calibration(pooled)
    plot_calibration_summary_v6(all_ece_raw, all_ece_cal,
                                  all_platt_w, all_platt_b, pooled)

    # ── Quality Bucket ────────────────────────────────────────────────
    plot_quality_bucket_full_v6(all_fold_metrics)

    # ── Patient-Level Analysis ────────────────────────────────────────
    fn_pts, fp_pts = plot_patient_level_analysis(all_fold_metrics)

    # ── K-Fold Summary ────────────────────────────────────────────────
    plot_kfold_summary_v6(all_fold_metrics, all_ece_cal,
                           all_platt_w, pooled)

    # ── Best Fold Grad-CAM ────────────────────────────────────────────
    aucs      = [m["auc"] for m in all_fold_metrics]
    best_fold = all_fold_metrics[np.argmax(aucs)]
    best_ckpt = Config.OUTPUT_DIR/"checkpoints"/f"fold_{best_fold['fold']}_best.pth"
    ckpt      = torch.load(best_ckpt, map_location=device, weights_only=False)
    best_model= HYGDModel(pretrained=False).to(device)
    best_model.load_state_dict(ckpt["model_state"])
    # Refit Platt untuk best fold
    best_va_df   = patient_folds[best_fold["fold"]-1][1]
    best_platt   = fit_platt_scaling(best_model, best_va_df, device)
    visualize_gradcam_v6(best_model, df_test, device,
                          platt_scaler=best_platt, n=8)

    # ── Final Summary ─────────────────────────────────────────────────
    accs  = [m["accuracy"]          for m in all_fold_metrics]
    senss = [m["sensitivity"]       for m in all_fold_metrics]
    specs = [m["specificity"]       for m in all_fold_metrics]
    pats  = [m["patient_accuracy"]  for m in all_fold_metrics]

    print("\n" + "="*72)
    print(f"  HASIL AKHIR v6 — {Config.N_FOLDS}-Fold CV + TTA + Platt Scaling")
    print("="*72)
    print(f"  {'Fold':<8} {'AUC':>8} {'Acc':>8} {'Sens':>8} "
          f"{'Spec':>8} {'ECE_raw':>9} {'ECE_cal':>9} "
          f"{'PatAcc':>8} {'w':>7} {'b':>7}")
    print(f"  {'-'*84}")
    for m, er, ec, w_, b_ in zip(all_fold_metrics, all_ece_raw, all_ece_cal,
                                   all_platt_w, all_platt_b):
        print(f"  Fold {m['fold']:<4} {m['auc']:>8.4f} {m['accuracy']:>8.4f} "
              f"{m['sensitivity']:>8.4f} {m['specificity']:>8.4f} "
              f"{er:>9.4f} {ec:>9.4f} "
              f"{m['patient_accuracy']:>8.4f} {w_:>7.3f} {b_:>7.3f}")
    print(f"  {'-'*84}")
    print(f"  {'Mean':<8} {np.mean(aucs):>8.4f} {np.mean(accs):>8.4f} "
          f"{np.mean(senss):>8.4f} {np.mean(specs):>8.4f} "
          f"{np.mean(all_ece_raw):>9.4f} {np.mean(all_ece_cal):>9.4f} "
          f"{np.mean(pats):>8.4f}")
    print(f"\n  Best Fold     : Fold {best_fold['fold']} (AUC={best_fold['auc']:.4f})")
    print(f"  Pooled ECE    : {pooled['ece_raw']:.4f} → {pooled['ece_cal']:.4f} "
          f"({'✓ <0.08' if pooled['ece_cal']<0.08 else '⚠ >0.08'} target)")
    print(f"  Patient Acc   : Mean={np.mean(pats):.4f} (majority vote)")
    print("="*72)

    # ── Summary JSON ──────────────────────────────────────────────────
    summary = {
        "version":   "v6",
        "n_folds":   Config.N_FOLDS,
        "architecture": {
            "backbone":      "EfficientNet-B3",
            "pretrained":    "ImageNet",
            "img_size":      Config.IMG_SIZE,
            "dual_head":     "cls + quality_regression",
            "calibration":   "Platt Scaling (LogisticRegression on logits)",
        },
        "training": {
            "loss":            f"FocalLoss(α={Config.FOCAL_ALPHA},γ={Config.FOCAL_GAMMA})+LabelSmoothing(ε={Config.LABEL_SMOOTHING})+MSE(λ={Config.LAMBDA_QUALITY})",
            "optimizer":       "AdamW layerwise LR",
            "lr_backbone":     Config.LR_BACKBONE,
            "lr_head":         Config.LR_HEAD,
            "weight_decay":    Config.WEIGHT_DECAY,
            "epochs_max":      Config.EPOCHS,
            "patience":        Config.PATIENCE,
            "batch_size":      Config.BATCH_SIZE,
        },
        "augmentation": {
            "train": ["RandomCrop","HFlip","VFlip","Rotate15","CLAHE(p=0.3)",
                      "ElasticTransform(α=0.5,p=0.2)","GridDistortion(p=0.2)",
                      "RandomGamma(80-120,p=0.3)[NEW v6]","ColorJitter","GaussNoise"],
            "tta":   ["HFlip","Rotate10","ColorJitter","RandomGamma(90-110,p=0.2)"],
            "tta_n": Config.TTA_N_AUGMENTS,
        },
        "data": {
            "quality_normalization":    "(score-1)/9",
            "quality_reject_threshold": Config.QUALITY_REJECT_THRESHOLD,
            "max_img_per_patient":      Config.MAX_IMG_PER_PATIENT,
            "split":                    "patient-level stratified 3-fold",
        },
        "results": {
            "mean_auc":                float(np.mean(aucs)),
            "std_auc":                 float(np.std(aucs)),
            "mean_accuracy":           float(np.mean(accs)),
            "mean_sensitivity":        float(np.mean(senss)),
            "mean_specificity":        float(np.mean(specs)),
            "mean_ece_raw":            float(np.mean(all_ece_raw)),
            "mean_ece_cal":            float(np.mean(all_ece_cal)),
            "pooled_ece_raw":          float(pooled["ece_raw"]),
            "pooled_ece_cal":          float(pooled["ece_cal"]),
            "pooled_brier":            float(pooled["brier"]),
            "mean_patient_accuracy":   float(np.mean(pats)),
            "best_fold":               int(best_fold["fold"]),
        },
        "platt_params": [{"fold": fi+1, "w": float(w), "b": float(b)}
                         for fi,(w,b) in enumerate(zip(all_platt_w, all_platt_b))],
        "changes_from_v5": [
            "REPLACE Temperature Scaling → Platt Scaling (2-param w,b)",
            "ADD RandomGamma(gamma_limit=(80,120), p=0.3) to train augmentation",
            "ADD RandomGamma(90-110, p=0.2) to TTA",
            "ADD Pooled ECE (3 fold, 5 bin) as primary calibration metric",
            "ADD Patient-level majority vote as primary clinical metric",
            "ADD README.md + requirements.txt auto-generation",
            "UPDATE Citation: HYGD v1.1.0 + GONet IEEE TBME 73(1):32-39, 2026",
        ],
        "citations": {
            "dataset": "Abramovich et al. (2026). HYGD v1.1.0. DOI:10.13026/m92s-0z95",
            "paper":   "Abramovich et al. (2026). GONet. IEEE TBME 73(1):32-39. DOI:10.1109/TBME.2025.3576688",
            "platform":"Goldberger et al. (2000). Circulation 101(23):e215-e220.",
        },
        "limitations": {
            "single_institution": "Hillel Yaffe Medical Center, Israel",
            "single_camera":      "TOPCON DRI OCT Triton, 45° FOV",
            "nationality":        "Israeli patients, age 36-95",
            "test_size":          f"Only {len(df_test)} images (28 GON-) — ECE estimate limited",
            "generalization":     "Cross-institution validation not performed",
            "ece_note":           f"ECE target <0.05 not achievable with 28 GON- samples; <0.08 is realistic",
        },
        "fold_results": [{k:v for k,v in m.items()
                          if k not in ["probs_raw","probs_cal","logits_raw",
                                       "labels","qualities","pids","preds",
                                       "patient_summary"]}
                         for m in all_fold_metrics],
    }
    json.dump(summary,
              open(Config.OUTPUT_DIR/"kfold_summary_v6.json","w"), indent=2)

    # [v6] Generate README + requirements
    generate_reproducibility_files(summary["results"])

    print(f"\n  Output: {Config.OUTPUT_DIR.resolve()}")
    print("""
══════════════════════════════════════════════════════════════════════
 CITATION WAJIB (IDSC 2026):
 [Dataset] Abramovich, O., Pizem, H., Fhima, J., Berkowitz, E.,
           Gofrit, B., Baskin, M., Meisel, M., Van Eijgen, J.,
           Blumenthal, E., & Behar, J. (2026). HYGD v1.1.0. PhysioNet.
           https://doi.org/10.13026/m92s-0z95
 [Paper]   Abramovich, O., et al. (2026). GONet: A Generalizable Deep
           Learning Model for Glaucoma Detection.
           IEEE Trans. Biomed. Eng., 73(1), 32-39.
           https://doi.org/10.1109/TBME.2025.3576688
 [Platform] Goldberger, A., et al. (2000). PhysioBank, PhysioToolkit,
            and PhysioNet. Circulation, 101(23), e215-e220.
══════════════════════════════════════════════════════════════════════
 LIMITASI DATASET (wajib di laporan):
 - Single institution : Hillel Yaffe Medical Center, Israel
 - Single camera      : TOPCON DRI OCT Triton, 45° FOV
 - Single nationality : Israeli, usia 36-95 tahun
 - Test set kecil     : 106 gambar (28 GON-) → ECE <0.05 tidak realistis
 - Generalisasi belum divalidasi ke klinik/kamera lain
══════════════════════════════════════════════════════════════════════
    """)


if __name__ == "__main__":
    main()